In [9]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

#df = pd.read_excel('POM PAID-PRIO_ARNG_workingpython.xlsx',sheet_name='Sum Cost')
df = pd.read_excel('POM PAID-PRIO_ARNG_workingpython.xlsx',sheet_name='Sum Driver')
df.columns = df.columns.str.strip()  # Removes leading/trailing spaces

In [10]:
# Define years and future forecast years
years = ["2019", "2020", "2021", "2022", "2023", "2024"]
forecast_years = [str(year) for year in range(2025, 2032)]  # Forecasting till 2030

# Create a new dataframe to store forecasts
df_forecast = df.copy()

In [11]:
# Run SARIMAX model for each row (each ID)
for index, row in df.iterrows():
    try:
        # Convert row data to numeric time series
        time_series = row[years].astype(float).values
        time_series = np.where(time_series > 0, time_series, 1)  # Avoid log(0)

        # Apply log transformation
        log_series = np.log(time_series)
        
        # Fit SARIMAX model
        model = SARIMAX(log_series, order=(1,1,1), seasonal_order=(1,1,1,3), enforce_stationarity=False, enforce_invertibility=False)
        fit = model.fit(disp=False)
        
        # Generate forecasts
        forecast_values_log = fit.forecast(steps=len(forecast_years))

        # Convert forecasts back from log scale
        forecast_values = np.exp(forecast_values_log)
        
        # Ensure non-negative forecasts
        forecast_values = np.maximum(forecast_values, 0)

        # Store forecasts in dataframe
        for i, year in enumerate(forecast_years):
            df_forecast.at[index, year] = forecast_values[i]
    
    except Exception as e:
        print(f"Skipping row {index} due to error: {e}")

c:\Users\john.voltz\Anaconda3\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
c:\Users\john.voltz\Anaconda3\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
c:\Users\john.voltz\Anaconda3\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
c:\Users\john.voltz\Anaconda3\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarnin

In [12]:
# Save forecasted data to a new Excel file
export_file_path = "SARIMAXDriver2.xlsx"
df_forecast.to_excel(export_file_path, index=False)
print(f"Forecasted data saved as {export_file_path}")

Forecasted data saved as SARIMAXDriver2.xlsx
